# 1. Project Introduction

The goal of this project is to build a Linear Regression model that predicts house prices based on different property features. We will explore the dataset, clean the data, preprocess features, train the model, and evaluate its performance.

# 2. Import Libraries

In [1]:
# Import Pandas for data manipulation and analysis
import pandas as pd

# Import NumPy for numerical calculations
import numpy as np

# Import functions for splitting the dataset
from sklearn.model_selection import train_test_split

# Import OneHotEncoder for categorical feature encoding
from sklearn.preprocessing import OneHotEncoder

# Import the Linear Regression model
from sklearn.linear_model import LinearRegression

# Import evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 3. Load the Dataset

In [2]:
# Load the house price dataset from the CSV file
price_data = pd.read_csv("HousePricePrediction.csv")

In [60]:
# Display the rows of the dataset
price_data.head()

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0


# 4. Exploratory Data Analysis

In [4]:
# Check the number of missing values in each column
price_data.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
Exterior1st        1
BsmtFinSF2         1
TotalBsmtSF        1
SalePrice       1459
dtype: int64

In [5]:
# Check the number of duplicate rows in the dataset
price_data.duplicated().sum()

np.int64(0)

### Explore Numerical Features

In [10]:
# Examine the statistical summary of numerical columns with missing values
price_data[["BsmtFinSF2","TotalBsmtSF"]].describe()

,BsmtFinSF2,TotalBsmtSF
count,2918.000000,2918.000000
mean,49.582248,1051.777587
std,169.205611,440.766258
min,0.000000,0.000000
25%,0.000000,793.000000
50%,0.000000,989.500000
75%,0.000000,1302.000000
max,1526.000000,6110.000000


# 5. Data Preprocessing

In [8]:
# Fill missing categorical values with the most frequent value (mode)
price_data["MSZoning"] = price_data["MSZoning"].fillna(price_data["MSZoning"].mode()[0])
price_data["Exterior1st"] = price_data["Exterior1st"].fillna(price_data["Exterior1st"].mode()[0])

In [9]:
# Verify that no missing values remain in the categorical columns
price_data[["MSZoning", "Exterior1st"]].isnull().sum()

np.int64(0)

In [12]:
# Fill missing numerical values with the median of their respective columns
price_data["BsmtFinSF2"] = price_data["BsmtFinSF2"].fillna(price_data["BsmtFinSF2"].median())
price_data["TotalBsmtSF"] = price_data["TotalBsmtSF"].fillna(price_data["TotalBsmtSF"].median())

In [59]:
# Separate the dataset into rows with known and missing SalePrice values
known_saleprice = price_data[price_data["SalePrice"].notnull()]
missing_saleprice = price_data[price_data["SalePrice"].isnull()]

# Display the shapes of both datasets
print(known_saleprice.shape)
print(missing_saleprice.shape)

(1460, 13)
(1459, 13)


In [7]:
# Check missing values in the dataset rows with unknown SalePrice
missing_saleprice.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
Exterior1st        1
BsmtFinSF2         1
TotalBsmtSF        1
SalePrice       1459
dtype: int64

In [13]:
# Verify that no missing values remain in the numerical columns
price_data[["BsmtFinSF2", "TotalBsmtSF"]].isnull().sum()

np.int64(0)

In [14]:
# Separate features (X) and target variable (y)
X = known_saleprice.drop(columns = ["SalePrice", "Id"])
y = known_saleprice["SalePrice"]

### Explore Categorical Features

In [17]:
# Identify the unique categories in the MSZoning column
price_data["MSZoning"].unique()

array(['RL', 'RM', 'C (all)', 'FV', 'RH'], dtype=object)

In [18]:
# Identify the unique categories in the LotConfig column
price_data["LotConfig"].unique()

array(['Inside', 'FR2', 'Corner', 'CulDSac', 'FR3'], dtype=object)

In [19]:
# Identify the unique categories in the BldgType column
price_data["BldgType"].unique()

array(['1Fam', '2fmCon', 'Duplex', 'TwnhsE', 'Twnhs'], dtype=object)

In [20]:
# Identify the unique categories in the Exterior1st column
price_data["Exterior1st"].unique()

array(['VinylSd', 'MetalSd', 'Wd Sdng', 'HdBoard', 'BrkFace', 'WdShing',
       'CemntBd', 'Plywood', 'AsbShng', 'Stucco', 'BrkComm', 'AsphShn',
       'Stone', 'ImStucc', 'CBlock'], dtype=object)

# 6. Train-Test Split

In [21]:
# Split the dataset into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 7. Feature Encoding

In [22]:
# Initialize the encoder
encoder = OneHotEncoder(drop = 'first', handle_unknown ='ignore')

# Identify categorical columns
categorical_columns = ['MSZoning', 'LotConfig', 'BldgType', 'Exterior1st']

# Fit the encoder on training data and transform training and testing data
encoder.fit(X_train[categorical_columns])
X_train_encoded = encoder.transform(X_train[categorical_columns])
X_test_encoded = encoder.transform(X_test[categorical_columns])

# Convert sparse matrices into NumPy arrays
X_train_encoded_toarray = X_train_encoded.toarray()
X_test_encoded_toarray = X_test_encoded.toarray()

# Get the names of the newly created encoded columns
encoded_columns = encoder.get_feature_names_out(categorical_columns)

# Create DataFrames for the encoded training and testing features
X_train_encoded_df = pd.DataFrame(
    X_train_encoded_toarray,
    columns=encoded_columns,
    index = X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded_toarray,
    columns = encoded_columns,
    index = X_test.index
)

# Identify and select numerical columns
numerical_columns = X_train.drop(columns = categorical_columns).columns

X_train_numerical = X_train[numerical_columns]
X_test_numerical = X_test[numerical_columns]

# Combine numerical and encoded categorical features
X_train_final = pd.concat([X_train_numerical, X_train_encoded_df], axis=1)
X_test_final = pd.concat([X_test_numerical, X_test_encoded_df], axis=1)

In [27]:
# Display the shapes of the datasets to verify the data split
print("Known SalePrice:", known_saleprice.shape)
print("Missing SalePrice:", missing_saleprice.shape)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Known SalePrice: (1460, 13)
Missing SalePrice: (1459, 13)
X shape: (1460, 11)
y shape: (1460,)
X_train shape: (1168, 11)
X_test shape: (292, 11)
y_train shape: (1168,)
y_test shape: (292,)


# 8. Model Training

In [33]:
# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train_final, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [34]:
# Predict house prices using the testing feature dataset
y_pred = model.predict(X_test_final)

In [36]:
# Display the first 10 predicted house prices
y_pred[:10]

array([168844.20187047, 243627.93266265, 114499.46802618, 157672.74928536,
       254809.72691561, 156499.8871421 , 177912.63550294, 127514.41651571,
       133133.47244362, 149402.60951095])

In [37]:
# Display the first 10 actual house prices from the testing dataset
y_test.head(10)

892     154500.0
1105    325000.0
413     115000.0
522     159000.0
1036    315500.0
614      75500.0
218     311500.0
1160    146000.0
649      84500.0
887     135500.0
Name: SalePrice, dtype: float64

# 9. Model Evaluation

In [38]:
# MAE (Mean Absolute Error)

# Calculate the average absolute difference between actual and predicted prices
mae = mean_absolute_error(y_test, y_pred)

# Display the Mean Absolute Error
print(mae)

34122.85443215001


In [39]:
# MSE (Mean Squared Error)

# Calculate the Mean Squared Error between actual and predicted prices
mse = mean_squared_error(y_test, y_pred)

# Display the Mean Squared Error
print(mse)

2918458029.585501


In [40]:
# RMSE (Root Mean Squared Error)

# Calculate the Root Mean Squared Error from MSE
rmse = np.sqrt(mse)

# Display the Root Mean Squared Error
print(rmse)

54022.754738957


In [41]:
# Predict house prices for the training dataset
y_train_pred = model.predict(X_train_final)

In [42]:
# R Squared

# Calculate the R-squared score on the testing dataset
r2 = r2_score(y_test, y_pred)
print("Testing R-squared:", r2)

# Calculate the R-squared score on the training dataset
r2_train = r2_score(y_train, y_train_pred)
print("Training R-squared:", r2_train)

r-squared :  0.6195131796544853
0.6142291549169858


# 10. Predicting Missing Sale Prices

In [44]:
# Extract the input features for the test dataset, excluding SalePrice and Id
X_missing = missing_saleprice.drop(columns = ["SalePrice", "Id"])

In [46]:
# Transform categorical features in the test dataset using the fitted encoder
X_missing_encoded = encoder.transform(X_missing[categorical_columns])

In [47]:
# Convert the encoded sparse matrix into a NumPy array
X_missing_encoded_toarray = X_missing_encoded.toarray()

In [48]:
# Create a DataFrame from the encoded categorical features
X_missing_encoded_df = pd.DataFrame(
    X_missing_encoded_toarray,
    columns=encoded_columns,
    index = X_missing.index
)

In [49]:
# Select numerical features from the test dataset
X_missing_numerical = X_missing[numerical_columns]

In [50]:
# Combine numerical and encoded categorical features into the final test dataset
X_missing_final = pd.concat([X_missing_numerical, X_missing_encoded_df], axis=1)

In [52]:
# Predict sale prices for houses with missing SalePrice values
missing_price_predictions = model.predict(X_missing_final)

In [61]:
# Create a dictionary containing house IDs and their predicted sale prices
df = {
    "Id" : missing_saleprice["Id"],
    "Predicted_SalePrice" : missing_price_predictions
}

# Convert the dictionary into a Pandas DataFrame
final_prediction = pd.DataFrame(df)

# Display the first five rows of the final prediction DataFrame
final_prediction.head()

,Id,Predicted_SalePrice
1460,1460,129857.642821
1461,1461,170773.275860
1462,1462,219012.851978
1463,1463,221292.646685
1464,1464,166757.424380


In [55]:
# Check the shape of the final prediction DataFrame
final_prediction.shape

(1459, 2)

In [56]:
# Check for missing values in the final prediction DataFrame
final_prediction.isnull().sum()

Id                     0
Predicted_SalePrice    0
dtype: int64

In [57]:
# Save the final predictions to a CSV file
final_prediction.to_csv(
    "Predicted_House_Prices.csv",
    index=False
)